# 03 - Despliegue

Este notebook valida que el artefacto desplegable sea un pipeline completo y que acepte datos crudos pre-carrera. La fase de Evaluación ya ocurrió antes; aquí se documenta despliegue y monitoreo.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / 'models' / 'best_model_pipe.pkl'
OUTPUT_DIR = ROOT / 'output'
pipe = joblib.load(MODEL_PATH)
print('Modelo cargado:', MODEL_PATH)
print('Pasos:', list(pipe.named_steps.keys()))

Modelo cargado: /home/creep/workshop/proyecto-mineria/models/best_model_pipe.pkl
Pasos: ['drop_leakage', 'feature_engineering', 'winsorizer', 'selector', 'sampler', 'scaler', 'model']


In [2]:
with open(OUTPUT_DIR / 'feature_schema.json', encoding='utf-8') as f:
    schema = json.load(f)
print(json.dumps(schema, indent=2, ensure_ascii=False))

{
  "target": "finished",
  "leakage_columns_removed": [
    "laps"
  ],
  "selected_features": [
    "grid",
    "driver_race_count",
    "driver_prev_finish_rate",
    "driver_last5_finish_rate",
    "constructor_race_count",
    "constructor_prev_finish_rate",
    "constructor_prev_avg_grid",
    "constructor_last5_finish_rate",
    "circuit_finish_rate",
    "circuit_avg_grid",
    "q1_seconds",
    "q2_seconds",
    "q3_seconds",
    "has_qualifying",
    "top10_start",
    "year",
    "round",
    "driver_nationality_encoded",
    "constructor_nationality_encoded",
    "circuit_country_encoded",
    "circuitRef_encoded",
    "experience_ratio",
    "grid_above_avg",
    "avg_finish_rate"
  ],
  "categorical_features_for_smotenc": [
    "has_qualifying",
    "top10_start",
    "driver_nationality_encoded",
    "constructor_nationality_encoded",
    "circuit_country_encoded",
    "circuitRef_encoded",
    "grid_above_avg"
  ],
  "smotenc_categorical_indices": [
    13,
    14,
    

In [3]:
raw_test = pd.read_csv(OUTPUT_DIR / 'test_unbalanced_raw.csv')
X_raw = raw_test.drop(columns=['finished'])
y_raw = raw_test['finished']

sample = X_raw.head(10)
pred = pipe.predict(sample)
prob = pipe.predict_proba(sample)[:, 1]
display(pd.DataFrame({'real': y_raw.head(10).to_numpy(), 'pred': pred, 'prob_finish': prob}).round(4))

,real,pred,prob_finish
0,0,0,0.3110
1,0,0,0.0254
2,0,0,0.2611
3,0,0,0.1903
4,0,0,0.3920
5,1,1,0.8647
6,0,0,0.1364
7,1,1,0.8668
8,1,1,0.7557
9,1,1,0.7575


In [4]:
# Ejemplo manual con columnas crudas pre-carrera. No se envían variables escaladas ni derivadas.
example = X_raw.median(numeric_only=True).to_frame().T
example['grid'] = 1
example['driver_race_count'] = 120
example['constructor_race_count'] = 350
example['driver_prev_finish_rate'] = 0.85
example['constructor_prev_finish_rate'] = 0.82
example['driver_last5_finish_rate'] = 0.80
example['constructor_last5_finish_rate'] = 0.80
example['has_qualifying'] = 1
example['top10_start'] = 1

pred = pipe.predict(example)[0]
prob = pipe.predict_proba(example)[0, 1]
print(f'Predicción ejemplo crudo: pred={pred}, prob_finish={prob:.4f}')
display(example.T.rename(columns={0: 'valor'}))

Predicción ejemplo crudo: pred=1, prob_finish=0.8823


,valor
grid,1.000000
driver_age,29.612594
driver_race_count,120.000000
driver_prev_finish_rate,0.850000
driver_last5_finish_rate,0.800000
constructor_race_count,350.000000
constructor_prev_finish_rate,0.820000
constructor_prev_avg_grid,10.460795
constructor_last5_finish_rate,0.800000
circuit_finish_rate,0.240546


## Streamlit

La aplicación `app/app.py` carga `models/best_model_pipe.pkl`. La interfaz debe enviar columnas crudas compatibles con `test_unbalanced_raw.csv`; el pipeline se encarga de ingeniería, selección, winsorización y escalado. En inferencia, el paso `sampler` no altera una muestra nueva porque `imblearn.Pipeline.predict` solo aplica transformaciones y el estimador final.